# 00 Preparación del equipo 8

Ejecutar una vez con el propietario del esquema y del volumen. Reutiliza el WPI, Bronze y su checkpoint. Crea la versión con control de agua, sus métricas y el esquema de consumo. El job diario empieza en 01.

In [ ]:
# Capturar los parámetros del notebook llamador antes de %run.
dbutils.widgets.text("fecha", "2023-06-08")
dbutils.widgets.text("ancla_replay", "")
dbutils.widgets.text("inicio_run", "")
ow_argumentos = tuple(dbutils.widgets.get(k).strip() for k in ("fecha", "ancla_replay", "inicio_run"))

In [ ]:
%run ./_config

## Volume y carpetas

`raw_zips/` guarda el zip, `landing/` el CSV que lee bronze, `checkpoints/` los
checkpoints de los streams, `ref/` el World Port Index y `_schemas/` el esquema de Auto Loader.


In [ ]:
import os

spark.sql(f"""
CREATE VOLUME IF NOT EXISTS {VOLUME_NAME}
COMMENT 'Archivos de OceanWatch: zips NOAA, CSV de landing, checkpoints y referencias.'
""")

for ruta in (RAW_ZIPS, LANDING, CHECKPOINTS, SCHEMAS, REF):
    os.makedirs(ruta, exist_ok=True)
    print("Listo", ruta)


## Tablas

Bronze conserva los mensajes y su procedencia. Silver v2 agrupa físicamente por fecha y conserva una observación válida por MMSI e instante. Las copias inválidas llegan a cuarentena con sus causas. La tabla original de métricas recibe el resumen de Bronze y `oceanwatch_metricas_silver_v2` registra el balance por microbatch. En el esquema de oro, 04 publica trayectorias, presencia portuaria y tráfico. La tarea 05 publica las evaluaciones de velocidad.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{ORO}` COMMENT 'Productos diarios de OceanWatch para consumo analitico.'")

tablas = {
    BRONZE: """
        (
          MMSI STRING,
          BaseDateTime TIMESTAMP,
          LAT DOUBLE,
          LON DOUBLE,
          SOG DOUBLE,
          COG DOUBLE,
          Heading DOUBLE,
          VesselName STRING,
          IMO STRING,
          CallSign STRING,
          VesselType INT,
          Status INT,
          Length DOUBLE,
          Width DOUBLE,
          Draft DOUBLE,
          Cargo STRING,
          TransceiverClass STRING,
          _source_file STRING COMMENT 'Ruta del CSV en landing.',
          _ingested_at TIMESTAMP COMMENT 'Instante en que el batch escribio la fila.'
        )
        USING DELTA
        COMMENT 'Bronze AIS: mensaje crudo, tal como llego. Cada archivo se lee una vez. La deduplicacion es en silver.'
    """,
    SILVER: """
        (
          MMSI STRING,
          BaseDateTime TIMESTAMP,
          LAT DOUBLE,
          LON DOUBLE,
          SOG DOUBLE,
          COG DOUBLE,
          Heading DOUBLE,
          VesselName STRING,
          IMO STRING,
          CallSign STRING,
          VesselType INT,
          Status INT,
          Length DOUBLE,
          Width DOUBLE,
          Draft DOUBLE,
          Cargo STRING,
          TransceiverClass STRING,
          _source_file STRING,
          _ingested_at TIMESTAMP,
          fecha_dia DATE,
          lote_alta BIGINT
        )
        USING DELTA
        CLUSTER BY (fecha_dia)
        COMMENT 'Silver AIS: una fila por (MMSI, BaseDateTime) en cada fecha_dia. La deduplicacion es aqui.'
    """,
    CUARENTENA: """
        (
          MMSI STRING,
          BaseDateTime TIMESTAMP,
          LAT DOUBLE,
          LON DOUBLE,
          SOG DOUBLE,
          COG DOUBLE,
          Heading DOUBLE,
          VesselName STRING,
          IMO STRING,
          CallSign STRING,
          VesselType INT,
          Status INT,
          Length DOUBLE,
          Width DOUBLE,
          Draft DOUBLE,
          Cargo STRING,
          TransceiverClass STRING,
          _source_file STRING,
          _ingested_at TIMESTAMP,
          lote_calidad BIGINT,
          motivo STRING COMMENT 'Reglas fallidas, separadas por coma.'
        )
        USING DELTA
        COMMENT 'Filas AIS que no entran a silver, con el motivo.'
    """,
    METRICAS: """
        (
          fecha DATE COMMENT 'Dia del parametro del job.',
          etapa STRING COMMENT 'bronze o silver.',
          filas_antes BIGINT COMMENT 'Conteo de la tabla bronze antes del stream.',
          filas_despues BIGINT COMMENT 'Conteo de la tabla bronze despues del stream.',
          filas_entrada BIGINT COMMENT 'Filas del batch de silver.',
          filas_validas BIGINT COMMENT 'Filas del batch que pasan las reglas.',
          filas_cuarentena BIGINT COMMENT 'Filas del batch que van a cuarentena.',
          ejecutado_en TIMESTAMP
        )
        USING DELTA
        COMMENT 'Conteos de cada ejecucion de bronze y de cada batch de silver.'
    """,
    PUERTOS: """
        (
          numero INT COMMENT 'World Port Index Number.',
          nombre STRING,
          pais STRING,
          latitud DOUBLE,
          longitud DOUBLE
        )
        USING DELTA
        COMMENT 'World Port Index (UpdatedPub150), el mismo catalogo de la Parte 1.'
    """,
    GOLD_TRAY: """
        (
          fecha DATE,
          MMSI STRING,
          n_posiciones BIGINT,
          inicio TIMESTAMP,
          fin TIMESTAMP,
          lat_inicio DOUBLE,
          lon_inicio DOUBLE,
          lat_fin DOUBLE,
          lon_fin DOUBLE,
          distancia_km DOUBLE COMMENT 'Suma de tramos haversine del dia, radio 6371 km.',
          sog_media DOUBLE
        )
        USING DELTA
        PARTITIONED BY (fecha)
        COMMENT 'Una trayectoria por buque y dia, calculada sobre silver.'
    """,
    GOLD_PUERTO: """
        (
          fecha DATE,
          MMSI STRING,
          numero_puerto INT,
          puerto STRING,
          pais STRING,
          n_posiciones BIGINT,
          inicio TIMESTAMP,
          fin TIMESTAMP
        )
        USING DELTA
        PARTITIONED BY (fecha)
        COMMENT 'Buque con al menos una posicion del dia junto a un puerto del WPI.'
    """,
    GOLD_TRAFICO: """
        (
          fecha DATE,
          hora INT COMMENT 'Hora UTC de BaseDateTime.',
          h3_cell STRING COMMENT 'Celda H3 resolucion 8, la misma de la Parte 1.',
          n_posiciones BIGINT,
          n_buques BIGINT
        )
        USING DELTA
        PARTITIONED BY (fecha)
        COMMENT 'Matriz de trafico celda H3 por hora, un dia.'
    """,
}

tablas.update({
    METRICAS_V2: """(lote BIGINT, fecha_job DATE, entrada BIGINT, nuevas BIGINT,
        duplicados_lote BIGINT, duplicados_previos BIGINT, cuarentena BIGINT, ejecutado_en TIMESTAMP)
        USING DELTA COMMENT 'Balance de filas por microbatch del checkpoint silver_ais_v2.'""",
    CONTROL_ORO: """(fecha DATE, lote_plata BIGINT, publicado_en TIMESTAMP)
        USING DELTA COMMENT 'Último lote de Silver publicado en los tres productos diarios.'""",
    GOLD_ANOM: """(fecha DATE, MMSI STRING, n_posiciones BIGINT, sog_media DOUBLE,
        velocidad_tramos DOUBLE, brecha DOUBLE, score DOUBLE, es_anomalia BOOLEAN, estado STRING,
        modelo_version STRING, run_id STRING) USING DELTA PARTITIONED BY (fecha)
        COMMENT 'Evaluación de velocidad por buque y día, cobertura y versión MLflow.'"""
})
tablas[CONTROL_ML] = """(fecha DATE, revision_oro STRING, publicado_en TIMESTAMP) USING DELTA
    COMMENT 'Versiones de los días de oro usados en entrenamiento e inferencia.'"""

for nombre, cuerpo in tablas.items():
    spark.sql(f"CREATE TABLE IF NOT EXISTS {nombre} {cuerpo}")
    print("Tabla", nombre)
spark.sql(f"ALTER TABLE {BRONZE} SET TBLPROPERTIES ('delta.appendOnly'='true')")

## World Port Index

La Parte 1 usó `UpdatedPub150.csv` (`Main Port Name`, `Country Code`, `Latitude`,
`Longitude`). El número de puerto en el CSV viene como `7950.0`: se lee como
double y se guarda como int. Spark no puede leer el archivo del repo (eso pide
ANY FILE), así que Python lo copia desde la carpeta Git a `ref/` y la lectura
es sobre `/Volumes`.


In [ ]:
from pathlib import Path
import shutil
from pyspark.sql import functions as F

destino = Path(REF) / "UpdatedPub150.csv"
destino.parent.mkdir(parents=True, exist_ok=True)

def csv_en_repo() -> Path | None:
    """CSV en la raiz del repo, al lado de src/, visto desde la carpeta Git."""
    try:
        ruta_nb = (
            dbutils.notebook.entry_point.getDbutils()
            .notebook()
            .getContext()
            .notebookPath()
            .get()
        )
    except Exception:
        return None
    bases = [ruta_nb]
    if not ruta_nb.startswith("/Workspace"):
        bases.append("/Workspace" + ruta_nb)
    # src/00_setup -> raiz del repo. Se prueban las dos formas del path.
    for base in bases:
        candidato = Path(base).parent.parent / "UpdatedPub150.csv"
        if candidato.is_file():
            return candidato
    return None

origen = csv_en_repo()
if origen is not None:
    shutil.copyfile(origen, destino)
    print("Copiado", origen, "->", destino)
elif not destino.is_file():
    raise FileNotFoundError(
        "No esta UpdatedPub150.csv en el repo ni en "
        f"{destino}. Abre el notebook desde la carpeta Git del repo."
    )
else:
    print("Se usa la copia ya presente en", destino)

# El numero viene como '7950.0'. String -> int falla. Double y luego int.
(
    spark.read.option("header", True).csv(str(destino))
    .select(
        F.expr("try_cast(`World Port Index Number` AS DOUBLE)").cast("int").alias("numero"),
        F.col("Main Port Name").alias("nombre"),
        F.col("Country Code").alias("pais"),
        F.expr("try_cast(Latitude AS DOUBLE)").alias("latitud"),
        F.expr("try_cast(Longitude AS DOUBLE)").alias("longitud"),
    )
    .where(F.col("latitud").isNotNull() & F.col("longitud").isNotNull() & F.col("numero").isNotNull())
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(PUERTOS)
)
spark.sql(
    f"COMMENT ON TABLE {PUERTOS} IS "
    "'World Port Index (UpdatedPub150), el mismo catalogo de la Parte 1.'"
)
print("Puertos", spark.table(PUERTOS).count())


## Referencia de agua

La máscara clasifica mar, lagos y cauces cartografiados sobre una rejilla de 0,005°. El contrato explica la incertidumbre en costa y canales estrechos.

In [ ]:
import hashlib
from pathlib import Path

nombre_mascara = "agua_ne_v1.csv.gz"
salida_mascara = Path(REF) / nombre_mascara
wpi_repo = csv_en_repo()
candidatos = [Path.cwd() / "reference" / nombre_mascara,
              Path.cwd().parent / "reference" / nombre_mascara]
if wpi_repo:
    candidatos.insert(0, wpi_repo.parent / "reference" / nombre_mascara)
for candidato in candidatos:
    if candidato.is_file():
        shutil.copyfile(candidato, salida_mascara)
        break
if not salida_mascara.is_file():
    raise FileNotFoundError(f"Carga reference/{nombre_mascara} en {salida_mascara} y repite esta celda.")
if hashlib.sha256(salida_mascara.read_bytes()).hexdigest() != "0e2b94d4485ddd5d736a71e4960872261bcff0175f50d13b5743ab637ff8ad36":
    raise ValueError("La máscara de agua no corresponde a la versión documentada.")
if not spark.catalog.tableExists(MASCARA):
    (spark.read.option("header", True).schema("tesela INT, clase INT, bits_hex STRING")
     .csv(str(salida_mascara)).write.format("delta").mode("errorifexists").saveAsTable(MASCARA))
n, claves = spark.table(MASCARA).selectExpr("count(*)", "count(distinct tesela)").first()
assert n == claves == 64800, "Referencia incompleta de agua/tierra"

## Permisos

El grupo `mine-group8` puede leer y escribir el volume, y leer y modificar las tablas.


In [ ]:
group = "`mine-group8`"
harvy = "`h.benitez@uniandes.edu.co`"
luis = "`l.galvan@uniandes.edu.co`"

spark.sql(f"GRANT READ VOLUME ON VOLUME {VOLUME_NAME} TO {group}")
spark.sql(f"GRANT WRITE VOLUME ON VOLUME {VOLUME_NAME} TO {group}")

spark.sql(f"GRANT READ VOLUME ON VOLUME {VOLUME_NAME} TO {harvy}")
spark.sql(f"GRANT WRITE VOLUME ON VOLUME {VOLUME_NAME} TO {harvy}")

spark.sql(f"GRANT READ VOLUME ON VOLUME {VOLUME_NAME} TO {luis}")
spark.sql(f"GRANT WRITE VOLUME ON VOLUME {VOLUME_NAME} TO {luis}")

for tabla in tablas:
    spark.sql(f"GRANT SELECT ON TABLE {tabla} TO {group}")
    spark.sql(f"GRANT MODIFY ON TABLE {tabla} TO {group}")
    spark.sql(f"GRANT SELECT ON TABLE {tabla} TO {harvy}")
    spark.sql(f"GRANT MODIFY ON TABLE {tabla} TO {harvy}")
    spark.sql(f"GRANT SELECT ON TABLE {tabla} TO {luis}")
    spark.sql(f"GRANT MODIFY ON TABLE {tabla} TO {luis}")
    print("GRANT", tabla)
for esquema in (SCHEMA, ORO):
    spark.sql(f"GRANT USE SCHEMA ON SCHEMA `{CATALOG}`.`{esquema}` TO {group}")
spark.sql(f"GRANT CREATE MODEL ON SCHEMA `{CATALOG}`.`{ORO}` TO {group}")
spark.sql(f"GRANT SELECT ON TABLE {MASCARA} TO {group}")